# Vector Store: Generative AI

Este notebook demonstra o fluxo minimo para ingerir um PDF local em um Vector Store e consultar a base com File Search.

Arquivo local utilizado para demonstração: `ai-agents-for-oracle-cloud-erp-br.pdf`.

Requisitos: 
- Ter permissão para chamar llms via API KEY
- Possuir uma base Vector Store previamente criada
- Ter um projeto ativo para usar API responses
- Preencher corretamente as credenciais no .env

In [ ]:
.env:
# OCI OpenAI-compatible endpoint
OCI_REGION=us-chicago-1
OCI_OPENAI_BASE_URL=https://inference.generativeai.us-chicago-1.oci.oraclecloud.com/openai/v1

# OCI Generative AI credentials
OCI_PROJECT_OCID=ocid1.generativeaiproject.
OCI_GENAI_API_KEY=sk-...

# Existing vector store 
OCI_FILE_SEARCH_VECTOR_STORE_ID=vs_ord_...


In [1]:
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(".env")

PDF_PATH = Path("ai-agents-for-oracle-cloud-erp-br.pdf")
VECTOR_STORE_ID = os.environ["OCI_FILE_SEARCH_VECTOR_STORE_ID"]
MODEL = "openai.gpt-oss-120b"
RUN_ID = f"demo_pdf_{int(time.time())}"

client = OpenAI(
    base_url=os.environ["OCI_OPENAI_BASE_URL"],
    api_key=os.environ["OCI_GENAI_API_KEY"],
    project=os.environ["OCI_PROJECT_OCID"],
)

print("Cliente criado")
print("PDF:", PDF_PATH)
print("Vector Store:", VECTOR_STORE_ID)
print("Run ID:", RUN_ID)

Cliente criado
PDF: ai-agents-for-oracle-cloud-erp-br.pdf
Vector Store: vs_ord_114wle3lf5ere4c41ieoa27tz04h2lc20hh1chumosjjn3f2
Run ID: demo_pdf_1781553440


## 1. Upload do PDF

A funcao `client.files.create(...)` envia o arquivo para o Files criando um id unico do arquivo.

In [2]:
upload_start = time.perf_counter()

with PDF_PATH.open("rb") as file:
    uploaded_file = client.files.create(file=file, purpose="user_data")

upload_seconds = time.perf_counter() - upload_start

print("File ID:", uploaded_file.id)
print(f"Tempo de upload: {upload_seconds:.1f}s")

File ID: file-ord-700026dd-6f9c-4acb-b1a3-1ab423e4a816
Tempo de upload: 2.9s


## 2. Iniciar ingestao no Vector Store

A funcao `client.vector_stores.files.create(...)` anexa o arquivo ao Vector Store e inicia a ingestao para que ele agora seja vetorizado junto a base.

In [3]:
vector_store_file = client.vector_stores.files.create(
    vector_store_id=VECTOR_STORE_ID,
    file_id=uploaded_file.id,
    attributes={"demo_run": RUN_ID, "document_name": PDF_PATH.name},
)

print("Vector Store File ID:", vector_store_file.id)
print("Status inicial:", vector_store_file.status)

Vector Store File ID: file-ord-700026dd-6f9c-4acb-b1a3-1ab423e4a816
Status inicial: queued


## 3. Acompanhar status da ingestao

A funcao `client.vector_stores.files.retrieve(...)` consulta o status ate o arquivo ficar pronto.

In [ ]:
terminal_statuses = {"completed", "failed", "cancelled", "expired"}
ingestion_start = time.perf_counter()

while True:
    status_file = client.vector_stores.files.retrieve(
        vector_store_id=VECTOR_STORE_ID,
        file_id=uploaded_file.id,
    )

    elapsed = time.perf_counter() - ingestion_start
    print(f"{elapsed:.1f}s - {status_file.status}")

    if status_file.status in terminal_statuses:
        break

    if elapsed > 900:
        raise TimeoutError("[Timeout] A ingestao nao terminou em 900s")

    time.sleep(5)

print(f"Status final: {status_file.status}")
print(f"Tempo de ingestao: {elapsed:.1f}s")

0.2s - in_progress
6.1s - in_progress
11.8s - in_progress
17.5s - in_progress
23.2s - in_progress
29.0s - in_progress
34.8s - in_progress
40.6s - in_progress
46.5s - in_progress
52.1s - in_progress
57.8s - in_progress
63.6s - in_progress
69.4s - in_progress
75.1s - in_progress
80.9s - in_progress
86.6s - in_progress
92.4s - in_progress
98.2s - completed
Status final: completed
Tempo de ingestao: 98.2s


## 4. Consultar a base

A funcao `client.responses.create(...)` usa a ferramenta `file_search` para responder com base no Vector Store.

In [5]:
response = client.responses.create(
    model=MODEL,
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [VECTOR_STORE_ID],
            "filters": {"type": "eq", "key": "demo_run", "value": RUN_ID},
        }
    ],
    input="Resuma em 5 bullets o conteudo do PDF ingerido nesta demo.",
)

print(response.output_text)

- **Oracle AI Agents transformam o ERP** – agentes de IA funcionam como assistentes digitais que automatizam tarefas manuais, aprendem com interações passadas e entregam respostas, recomendações e execuções contextuais em processos financeiros e comerciais.  
- **Finanças orientadas por IA (Finance‑AI)** – a estratégia combina IA generativa, machine‑learning e automação para oferecer três pilares: operações sem intervenção humana, insights preditivos e ações conectadas, elevando eficiência, reduzindo erros e possibilitando decisões em tempo real.  
- **Três agentes‑chave no Oracle Cloud ERP** – (1) *Agente de Entrada/Saída de Documentos*: captura, padroniza e converte documentos (PDF, imagens, diferentes idiomas) em requisições, faturas ou instruções de pagamento; (2) *Agente do Livro‑Razo*: identifica exceções, anomalias e gera lançamentos contábeis corretivos via linguagem natural; (3) *Agente de Previsão Avançada*: cria previsões multivariadas (receita, caixa) usando dados internos,